# Gemini 를 OpenAI 라이브러리로 부르기

앞 노트북(`1_1_openai`)과 **거의 같은 코드**입니다. 그런데 답하는 건 구글 Gemini 입니다.

어떻게 가능할까요? 많은 LLM 회사가 **OpenAI 와 같은 모양의 API**를 함께 제공하기 때문입니다.
이것을 'OpenAI 호환(compatible) 엔드포인트' 라고 부릅니다.

## 1_1 과 무엇이 다른가

| | `1_1_openai` | 이 노트북 |
|---|---|---|
| 라이브러리 | `openai` | `openai` **(같다)** |
| 호출 코드 | `chat.completions.create(...)` | **(같다)** |
| `api_key` | OpenAI 키 | 구글 키 |
| `base_url` | 기본값 (생략) | **구글 주소** |
| `model` | `gpt-...` | `gemini-2.5-flash` |

> **바뀐 것은 딱 세 줄입니다.** 나머지 코드는 그대로입니다.
> 이것이 이 노트북의 전부이자 핵심입니다.

## Step 1. 가져오는 것은 그대로

`google` 패키지가 아니라 **`openai` 패키지를 그대로** 씁니다.
구글 모델을 쓰는데 OpenAI 라이브러리를 가져오는 것이 어색해 보이지만, 이게 요점입니다.

> 구글 전용 라이브러리(`google-genai`)도 있습니다. 기능을 모두 쓰려면 그쪽이 낫습니다.
> 여기서는 **여러 회사 모델을 같은 코드로 다루는 방법**을 배우는 것이 목적입니다.

In [1]:
import os
#import requests
from dotenv import load_dotenv
from openai import OpenAI

## Step 2. `.env` 읽기

`1_1` 과 똑같습니다. `True` 가 나오면 `.env` 파일을 제대로 찾았다는 뜻입니다.

In [2]:
load_dotenv(override=True)

False

## Step 3. 구글 API 키 확인

`OPENAI_API_KEY` 가 아니라 **`GOOGLE_API_KEY`** 를 읽습니다.
키는 [Google AI Studio](https://aistudio.google.com/apikey) 에서 발급받아 `.env` 에 적어 둡니다.

```
GOOGLE_API_KEY=AI...
```

> 메시지에 `(and this is optional)` 이 붙은 이유는
> 이 교육 과정에서 Gemini 가 필수가 아니기 때문입니다. 없으면 이 노트북만 건너뛰면 됩니다.

In [3]:
google_api_key = os.getenv('GOOGLE_API_KEY')
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

Google API Key exists and begins AQ


## Step 4. 주소(base_url) — 이 노트북의 핵심

```
https://generativelanguage.googleapis.com/v1beta/openai/
```

주소 끝에 **`/openai/`** 가 붙어 있는 것을 보세요.
"OpenAI 와 같은 모양으로 받아 주는 입구"라는 뜻입니다.

| 회사 | base_url |
|---|---|
| OpenAI | (생략 — 라이브러리 기본값) |
| Google | `https://generativelanguage.googleapis.com/v1beta/openai/` |
| Ollama (내 PC) | `http://localhost:11434/v1` |

주소도 `.env` 에 넣어 두었습니다. 코드에 박아 두면 바뀔 때마다 코드를 고쳐야 하기 때문입니다.

In [4]:
gemini_url = os.getenv('GEMINI_URL')
print(gemini_url)

https://generativelanguage.googleapis.com/v1beta/openai/


## Step 5. 클라이언트 만들기 — 두 줄이 전부

```python
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
```

`1_1` 에서는 `OpenAI()` 로 비워 뒀습니다. 환경변수에서 OpenAI 키를 자동으로 찾기 때문입니다.
여기서는 **다른 회사**를 쓰므로 키와 주소를 직접 알려 줘야 합니다.

변수 이름을 `gemini` 로 지었지만 **타입은 여전히 `OpenAI`** 입니다.
껍데기는 OpenAI 라이브러리이고, 안에서 말을 거는 상대만 구글로 바뀐 것입니다.

In [5]:
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

## Step 6. `system` 프롬프트

`1_1_openai` 와 **글자 하나까지 똑같은** 프롬프트를 씁니다.
같은 질문을 다른 모델에 던져 **답이 어떻게 다른지 비교**하려는 것입니다.

In [6]:
system_prompt = """
당신은 IT 보안 전문가입니다. 
시스템 보안에 대한 전문 지식을 제공하며 새롭게 발생하는 보안 문제에 대해서도 대처 방법을 제공합니다
"""

## Step 7. `user` 프롬프트

이것도 `1_1` 과 같습니다. 비교가 목적이므로 조건을 똑같이 맞춥니다.

In [7]:
user_prompt = """
나는 IT 회사의 보안 책임자입니다.
현재 가장 대두되고 있는 보안 이슈는 무엇이 있나요?

"""

## Step 8. 호출 — 모델 이름만 바뀐다

```python
response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=message)
```

`1_1` 의 호출과 비교해 보세요.

```python
# 1_1
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
# 여기
response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=message)
```

**클라이언트 변수와 모델 이름만 다릅니다.** 함수도, 인자 이름도, `messages` 구조도 같습니다.

| 모델 이름 | 성격 |
|---|---|
| `gemini-2.5-flash` | 빠르고 싸다. 일상적인 작업용 |
| `gemini-2.5-pro` | 느리고 비싸다. 어려운 추론용 |

In [8]:
message = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

response = gemini.chat.completions.create(model="gemini-3.8-flash", messages=message)


## Step 9. 응답 구조도 같은 모양

```python
print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))
```

구글이 답했는데도 **`choices[0].message.content`, `usage`, `finish_reason`** 이 그대로 있습니다.
OpenAI 호환이라는 말은 **응답 모양까지 같다**는 뜻입니다.

덕분에 응답을 처리하는 코드를 모델마다 새로 짜지 않아도 됩니다.

> `id` 값은 OpenAI 의 `chatcmpl-...` 과 형식이 다릅니다.
> 호환이라 해도 내부 값까지 같지는 않습니다.

In [13]:
import json

print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

{
  "id": "LIrMarHwNuyM1e8PkqvR-AI",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "IT 회사의 보안 책임자로서 회사의 비즈니스 연속성과 데이터 자산을 보호하는 중책을 맡고 계시군요.\n\n현재 글로벌 사이버 보안 환경은 생성형 AI의 도입, 클라우드 네이티브 환경의 확산, 그리고 복잡해진 소프트웨어 공급망으로 인해 그 어느 때보다 빠르게 변화하고 있습니다. 현재 엔터프라이즈 및 IT 기업 관점에서 **가장 대두되고 있는 핵심 보안 이슈 5가지**와 이에 따른 **대응 전략**을 정리해 드립니다.\n\n---\n\n### 1. 생성형 AI 관련 보안 위협 (AI-Driven Threats & AI Security)\n*가장 뜨거운 감자이며, 공격과 방어 양쪽 모두에서 패러다임 변화가 일어나고 있습니다.*\n* **이슈 내용:**\n  * **공격의 고도화:** 공격자들이 생성형 AI를 활용해 완벽한 현지어 피싱 메일, 우회형 악성코드, 딥페이크를 통한 CEO 사칭(BEC 공격)을 대규모로 자동화하고 있습니다.\n  * **사내 AI 도입의 취약점:** 기업 내 LLM 도입 과정에서 발생하는 **프롬프트 인젝션(Prompt Injection)**, 비공개 소스코드 및 개인정보 등 **민감 데이터의 LLM 학습 유출**, 안전하지 않은 AI 플러그인 연동 문제가 급증하고 있습니다.\n* **대응 방향:** \n  * 사내 생성형 AI 사용 가이드라인 및 DLP(데이터 유출 방지) 정책 수립.\n  * OWASP Top 10 for LLM을 기반으로 사내 AI 서비스의 보안성 검증(AI Red Teaming).\n\n### 2. 소프트웨어 공급망 공격 (Software Supply Chain Attacks)\n*IT 및 소프트웨어 개발사에게 가장 치

## Step 10. 답변 꺼내기

```python
print(response.choices[0].message.content)
```

`1_1` 과 똑같은 한 줄입니다.

Gemini 의 답을 `1_1_openai` 의 답과 비교해 보세요. 같은 질문인데 **구성과 깊이가 다릅니다.**
어느 쪽이 낫다기보다 모델마다 성격이 다르다고 보는 편이 맞습니다.
그래서 실무에서는 **용도별로 모델을 골라 씁니다.**

In [15]:
print(response.choices[0].message.content)

IT 회사의 보안 책임자로서 회사의 비즈니스 연속성과 데이터 자산을 보호하는 중책을 맡고 계시군요.

현재 글로벌 사이버 보안 환경은 생성형 AI의 도입, 클라우드 네이티브 환경의 확산, 그리고 복잡해진 소프트웨어 공급망으로 인해 그 어느 때보다 빠르게 변화하고 있습니다. 현재 엔터프라이즈 및 IT 기업 관점에서 **가장 대두되고 있는 핵심 보안 이슈 5가지**와 이에 따른 **대응 전략**을 정리해 드립니다.

---

### 1. 생성형 AI 관련 보안 위협 (AI-Driven Threats & AI Security)
*가장 뜨거운 감자이며, 공격과 방어 양쪽 모두에서 패러다임 변화가 일어나고 있습니다.*
* **이슈 내용:**
  * **공격의 고도화:** 공격자들이 생성형 AI를 활용해 완벽한 현지어 피싱 메일, 우회형 악성코드, 딥페이크를 통한 CEO 사칭(BEC 공격)을 대규모로 자동화하고 있습니다.
  * **사내 AI 도입의 취약점:** 기업 내 LLM 도입 과정에서 발생하는 **프롬프트 인젝션(Prompt Injection)**, 비공개 소스코드 및 개인정보 등 **민감 데이터의 LLM 학습 유출**, 안전하지 않은 AI 플러그인 연동 문제가 급증하고 있습니다.
* **대응 방향:** 
  * 사내 생성형 AI 사용 가이드라인 및 DLP(데이터 유출 방지) 정책 수립.
  * OWASP Top 10 for LLM을 기반으로 사내 AI 서비스의 보안성 검증(AI Red Teaming).

### 2. 소프트웨어 공급망 공격 (Software Supply Chain Attacks)
*IT 및 소프트웨어 개발사에게 가장 치명적인 위협입니다.*
* **이슈 내용:**
  * 기업 자체의 방어선보다 상대적으로 취약한 **서드파티 라이브러리, 오픈소스 컴포넌트, CI/CD 파이프라인**을 타깃으로 합니다.
  * 최근의 **'XZ Utils 백도어 사태'**처럼 오픈소스 생태계에 장기적으로 침투해 합법적인 업데이트 배포망을 통해 악성코드를 유포하는 고도화

---

## 정리

```python
OpenAI(api_key=..., base_url=...)   # 이 두 인자가 전부
```

| | 내용 |
|---|---|
| **라이브러리는 하나로** | `openai` 하나로 여러 회사 모델을 부를 수 있다 |
| **바뀌는 건 세 가지** | `api_key` · `base_url` · `model` |
| **응답 모양도 같다** | 처리 코드를 다시 짤 필요가 없다 |

### 직접 해 보기

1. `model` 을 `gemini-2.5-pro` 로 바꿔 보세요. 답과 `usage` 가 어떻게 달라지나요?
2. `base_url` 을 일부러 틀리게 적어 보세요. 어떤 오류가 나나요?
3. `1_1_openai` 의 답과 나란히 놓고 비교해 보세요.

### 다음 노트북

| 노트북 | 내용 |
|---|---|
| `1_3_ollama` | 같은 방법으로 **내 PC 안에서 도는** 모델 부르기 |
| `1_4_llms` | 네 가지 모델에 같은 질문을 던져 한 번에 비교 |